In [67]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

In [2]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b, this tool returns their product"""
    return a*b

In [3]:
multiply.invoke({'a': 2, 'b':10})

20

In [4]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [7]:
llm_with_tools = model.bind_tools([multiply])

In [10]:
result = llm_with_tools.invoke('do multiplication of 24324 and 5345').tool_calls

In [12]:
result[0]['args']

{'a': 24324, 'b': 5345}

In [ ]:
# only result
multiply.invoke(result[0]['args'])

130011780

In [ ]:
# result inside the toolmessage
multiply.invoke(result[0])

ToolMessage(content='130011780', name='multiply', tool_call_id='5ax60j1k8')

#### entire in one go

In [61]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

model = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [62]:
model_with_tools = model.bind_tools([multiply])

In [68]:
system_instruction = SystemMessage(
    "You are a helpful assistant. "
    "Use the provided tools to answer the user's question. "
    "Once you have the result from a tool, DO NOT call the tool again. "
    "Instead, explain the answer to the user in a natural sentence."
)

query = "can you multi of 2 and 3"

messages = [system_instruction]

human_query = HumanMessage(query)

messages.append(human_query)

response = model_with_tools.invoke(messages)

messages.append(response)

tool_message = multiply.invoke(response.tool_calls[0])

messages.append(tool_message)

result = model_with_tools.invoke(messages)

In [65]:
messages

[HumanMessage(content='can you multi of 2 and 3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bqx6vf2wr', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 241, 'total_tokens': 260, 'completion_time': 0.032560421, 'completion_tokens_details': None, 'prompt_time': 0.013821828, 'prompt_tokens_details': None, 'queue_time': 0.055484441, 'total_time': 0.046382249}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b643f-a910-75d3-90c5-cfa6855001a8-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'bqx6vf2wr', 'type': 'tool_call'}], usage_metadata={'input_tokens': 241, 'output_tokens': 19, 'total_tokens': 260}),
 ToolMessage(content='6', name='multiply

In [69]:
result

AIMessage(content='The product of 2 and 3 is 6.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 317, 'total_tokens': 330, 'completion_time': 0.015249146, 'completion_tokens_details': None, 'prompt_time': 0.020082624, 'prompt_tokens_details': None, 'queue_time': 0.055063365, 'total_time': 0.03533177}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b6452-b4d9-7471-bb55-15b33e387195-0', usage_metadata={'input_tokens': 317, 'output_tokens': 13, 'total_tokens': 330})